# P10 - Final ABSA Pipeline (Office_Products only)

Chạy pipeline cuối cho **Office_Products**: **P2 cleaned sentences → Gate → ATE → ASC → thống kê/report**.

Notebook này tự tải model/data bằng Google Drive ID:
- Gate model
- ATE phase 2
- ASC phase 3
- Data P2 Office_Products

Đầu ra:
- `outputs/p10/final_predictions_office_products.parquet`
- `outputs/p10/aspect_sentiment_counts_office_products.csv`
- `outputs/p10/neutral_low_conf_examples_office_products.txt`
- `outputs/reports/p10/p10_report_office_products.txt`


In [8]:
# ============================================================
# 0. INSTALL + IMPORT
# ============================================================
!pip install -q transformers accelerate pyarrow pandas numpy tqdm gdown

from google.colab import drive

from pathlib import Path
from collections import Counter, defaultdict
import os
import re
import json
import math
import shutil
import subprocess
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import gdown
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)


DEVICE = cuda


In [ ]:
# ============================================================
# 1. CONFIG + DOWNLOAD BY GOOGLE DRIVE ID - OFFICE_PRODUCTS ONLY
# ============================================================

# Google Drive IDs bạn cung cấp
GATE_MODEL_ID = '1YCF7tB8Waajw2o2C_d6C4eoOIhjzFio0'
ATE_MODEL_ID  = '1ERpB4Nxk5wFfCnuAEXihUQXotf12b27Y'
ASC_MODEL_ID  = '1sCsPdONQJa-p3yiGPYgVNWsaWL9x8FmT'
OFFICE_DATA_ID = '198j5Y-Ng7g16YQR7Y91mVelSd-4a4-6w'

CATEGORY_NAME = 'office_products'

ASSET_DIR = Path('/content/p10_assets')
ASSET_DIR.mkdir(parents=True, exist_ok=True)

GATE_MODEL_DIR = ASSET_DIR / 'gate_model'
ATE_MODEL_DIR  = ASSET_DIR / 'ate_phase2'
ASC_MODEL_DIR  = ASSET_DIR / 'asc_phase3'
OFFICE_DATA_DIR = ASSET_DIR / 'office_products_p2'
OFFICE_DATA_FILE = ASSET_DIR / 'office_products_p2.parquet'

# Nếu muốn tải lại từ đầu thì đổi thành True
FORCE_DOWNLOAD = False


def _has_files(path: Path):
    return path.exists() and any(path.iterdir()) if path.is_dir() else path.exists()


def download_gdrive_folder(folder_id: str, output_dir: Path, force: bool = False):
    output_dir = Path(output_dir)
    if force and output_dir.exists():
        shutil.rmtree(output_dir)

    if _has_files(output_dir):
        print(f'[EXISTS] {output_dir}')
        return output_dir

    output_dir.mkdir(parents=True, exist_ok=True)
    url = f'https://drive.google.com/drive/folders/{folder_id}'
    print(f'[DOWNLOAD FOLDER] {url} -> {output_dir}')
    gdown.download_folder(url, output=str(output_dir), quiet=False, use_cookies=False)
    return output_dir


def download_gdrive_file(file_id: str, output_path: Path, force: bool = False):
    output_path = Path(output_path)
    if force and output_path.exists():
        output_path.unlink()

    if output_path.exists() and output_path.stat().st_size > 0:
        print(f'[EXISTS] {output_path}')
        return output_path

    output_path.parent.mkdir(parents=True, exist_ok=True)
    url = f'https://drive.google.com/uc?id={file_id}'
    print(f'[DOWNLOAD FILE] {url} -> {output_path}')
    gdown.download(url, str(output_path), quiet=False, fuzzy=True)
    return output_path


def download_data_id_auto(data_id: str, folder_output: Path, file_output: Path, force: bool = False):
    """
    Data Office_Products có thể là folder parquet hoặc 1 file parquet.
    Hàm này ưu tiên tải folder. Nếu folder fail thì tải file.
    """
    try:
        folder_output = download_gdrive_folder(data_id, folder_output, force=force)
        parquet_files = list(folder_output.rglob('*.parquet'))
        if parquet_files:
            print(f'[DATA] Found {len(parquet_files)} parquet files in folder.')
            return folder_output
        print('[WARN] Folder downloaded but no parquet found. Try file download...')
    except Exception as e:
        print('[WARN] download_folder failed, try download file:', repr(e))

    download_gdrive_file(data_id, file_output, force=force)
    return file_output


# Tải model và data
download_gdrive_folder(GATE_MODEL_ID, GATE_MODEL_DIR, force=FORCE_DOWNLOAD)
download_gdrive_folder(ATE_MODEL_ID, ATE_MODEL_DIR, force=FORCE_DOWNLOAD)
download_gdrive_folder(ASC_MODEL_ID, ASC_MODEL_DIR, force=FORCE_DOWNLOAD)
OFFICE_DATA_PATH = download_data_id_auto(OFFICE_DATA_ID, OFFICE_DATA_DIR, OFFICE_DATA_FILE, force=FORCE_DOWNLOAD)

# Chỉ chạy Office_Products
P2_DATA_PATHS = {
    CATEGORY_NAME: Path(OFFICE_DATA_PATH),
}

OUTPUT_DIR = Path('/content/outputs/p10')
REPORT_DIR = Path('/content/outputs/reports/p10')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Threshold theo yêu cầu P10
GATE_THRESHOLD = 0.70
SPAN_THRESHOLD = 0.50
# ASC không dùng threshold, chỉ argmax.

BATCH_SIZE_GATE = 128
BATCH_SIZE_ATE  = 64
BATCH_SIZE_ASC  = 128
MAX_LEN_GATE = 192
MAX_LEN_ATE  = 192
MAX_LEN_ASC  = 192

# pyarrow batch size khi đọc parquet lớn
PARQUET_BATCH_ROWS = 50_000

LABEL_NAMES = {0: 'neg', 1: 'neu', 2: 'pos'}

# ============================================================
# MULTI-ACCOUNT BATCH RANGE
# ============================================================
# Chỉ chạy batch từ START_BATCH đến END_BATCH.
# Ví dụ acc này chạy 132-160 thì để như dưới.
# Muốn chạy tất cả batch thì đặt START_BATCH = 0, END_BATCH = 999999.
START_BATCH = 0
END_BATCH = 999

print('Batch range to run:', START_BATCH, '->', END_BATCH)


print('Config loaded')
print('GATE_MODEL_DIR =', GATE_MODEL_DIR)
print('ATE_MODEL_DIR  =', ATE_MODEL_DIR)
print('ASC_MODEL_DIR  =', ASC_MODEL_DIR)
print('P2_DATA_PATHS  =', P2_DATA_PATHS)


# Batch checkpoint: lưu từng batch lên Drive để Colab ngắt vẫn resume được.
# Nếu không mount Drive được thì sẽ lưu ở /content, nhưng nên mount Drive để an toàn.
try:
    drive.mount('/content/drive')
    DRIVE_AVAILABLE = True
except Exception as e:
    print('[WARN] Không mount được Drive, checkpoint chỉ lưu local /content:', repr(e))
    DRIVE_AVAILABLE = False

if DRIVE_AVAILABLE:
    P10_BATCH_DIR = Path('/content/drive/MyDrive/p10_office_batch_checkpoints')
    P10_FINAL_BACKUP_DIR = Path('/content/drive/MyDrive/p10_office_final_outputs')
else:
    P10_BATCH_DIR = Path('/content/p10_office_batch_checkpoints')
    P10_FINAL_BACKUP_DIR = Path('/content/p10_office_final_outputs')

P10_BATCH_DIR.mkdir(parents=True, exist_ok=True)
P10_FINAL_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# Nếu muốn chạy lại từ đầu, đặt True để bỏ qua checkpoint cũ.
RESET_BATCH_CHECKPOINTS = False
if RESET_BATCH_CHECKPOINTS and P10_BATCH_DIR.exists():
    shutil.rmtree(P10_BATCH_DIR)
    P10_BATCH_DIR.mkdir(parents=True, exist_ok=True)

print('Batch checkpoint dir =', P10_BATCH_DIR)
print('Final backup dir =', P10_FINAL_BACKUP_DIR)


[EXISTS] /content/p10_assets/gate_model
[EXISTS] /content/p10_assets/ate_phase2
[EXISTS] /content/p10_assets/asc_phase3
[EXISTS] /content/p10_assets/office_products_p2
[DATA] Found 4 parquet files in folder.
Batch range to run: 190 -> 999
Config loaded
GATE_MODEL_DIR = /content/p10_assets/gate_model
ATE_MODEL_DIR  = /content/p10_assets/ate_phase2
ASC_MODEL_DIR  = /content/p10_assets/asc_phase3
P2_DATA_PATHS  = {'office_products': PosixPath('/content/p10_assets/office_products_p2')}
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Batch checkpoint dir = /content/drive/MyDrive/p10_office_batch_checkpoints
Final backup dir = /content/drive/MyDrive/p10_office_final_outputs


In [13]:

# ============================================================
# 2. LOAD MODELS
# ============================================================

import json
import re
from pathlib import Path

print('Using fixed dirs:')
print('GATE_MODEL_DIR:', GATE_MODEL_DIR)
print('ATE_MODEL_DIR :', ATE_MODEL_DIR)
ASC_MODEL_DIR = ASC_MODEL_DIR / 'model'
print('ASC_MODEL_DIR :', ASC_MODEL_DIR)

# Gate
gate_tokenizer = AutoTokenizer.from_pretrained(str(GATE_MODEL_DIR), use_fast=True)
gate_model = AutoModelForSequenceClassification.from_pretrained(str(GATE_MODEL_DIR)).to(DEVICE)
gate_model.eval()

# ATE
ate_tokenizer = AutoTokenizer.from_pretrained(str(ATE_MODEL_DIR), use_fast=True)
ate_model = AutoModelForTokenClassification.from_pretrained(str(ATE_MODEL_DIR)).to(DEVICE)
ate_model.eval()

asc_tokenizer = AutoTokenizer.from_pretrained(str(ASC_MODEL_DIR), use_fast=True)
asc_model = AutoModelForSequenceClassification.from_pretrained(str(ASC_MODEL_DIR)).to(DEVICE)
asc_model.eval()

print('Loaded all models')
print('ATE id2label:', ate_model.config.id2label)
print('ASC id2label:', asc_model.config.id2label)

Using fixed dirs:
GATE_MODEL_DIR: /content/p10_assets/gate_model
ATE_MODEL_DIR : /content/p10_assets/ate_phase2
ASC_MODEL_DIR : /content/p10_assets/asc_phase3/model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded all models
ATE id2label: {0: 'O', 1: 'B-ASP', 2: 'I-ASP'}
ASC id2label: {0: 'negative', 1: 'neutral', 2: 'positive'}


In [14]:

# ============================================================
# 3. UTILS
# ============================================================

def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def mark_aspect(sentence, aspect):
    sentence = clean_text(sentence)
    aspect = clean_text(aspect)
    if not sentence or not aspect:
        return sentence
    pattern = re.compile(re.escape(aspect), flags=re.IGNORECASE)
    if pattern.search(sentence):
        return pattern.sub(f'[ASP] {aspect} [/ASP]', sentence, count=1)
    # fallback nếu aspect không match lại được trong câu
    return f'{sentence} [ASP] {aspect} [/ASP]'


def list_parquet_files(path: Path):
    path = Path(path)
    if path.is_file() and path.suffix == '.parquet':
        return [path]
    if path.is_dir():
        return sorted(path.rglob('*.parquet'))
    raise FileNotFoundError(f'Không thấy parquet path: {path}')


def iter_parquet_batches(path: Path, batch_rows=50_000):
    """Yield pandas DataFrame batches từ file/folder parquet."""
    files = list_parquet_files(path)
    if not files:
        raise FileNotFoundError(f'Không tìm thấy file parquet trong: {path}')
    for f in files:
        pf = pq.ParquetFile(str(f))
        for batch in pf.iter_batches(batch_size=batch_rows):
            yield batch.to_pandas()


def safe_col(df, col, default=None):
    if col in df.columns:
        return df[col]
    return pd.Series([default] * len(df), index=df.index)


In [15]:

# ============================================================
# 4. PREDICT FUNCTIONS: GATE -> ATE -> ASC
# ============================================================

@torch.no_grad()
def predict_gate(sentences, batch_size=BATCH_SIZE_GATE):
    probs_has = []
    sentences = [clean_text(x) for x in sentences]

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        enc = gate_tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=MAX_LEN_GATE,
            return_tensors='pt'
        ).to(DEVICE)

        logits = gate_model(**enc).logits

        # Nếu binary 2 logits: lấy prob class 1 = has_aspect.
        # Nếu 1 logit: sigmoid.
        if logits.shape[-1] == 1:
            prob = torch.sigmoid(logits[:, 0])
        else:
            prob = torch.softmax(logits, dim=-1)[:, 1]

        probs_has.extend(prob.detach().cpu().numpy().tolist())
    return probs_has


def _is_aspect_label(label_name, label_id):
    """Nhận diện label ATE là aspect hay không, hỗ trợ BIO hoặc binary."""
    s = str(label_name).upper()
    if 'ASP' in s or s in {'B', 'I', 'B-ASP', 'I-ASP', 'B-ASPECT', 'I-ASPECT'}:
        return True
    # fallback cho binary token classification: label 1 là aspect
    if str(label_name) in {'1', 'LABEL_1'} or int(label_id) == 1:
        return True
    return False


@torch.no_grad()
def extract_aspects_batch(sentences, batch_size=BATCH_SIZE_ATE):
    """Trích aspect bằng ATE token classification. Return list[(aspects, confidences)]."""
    outputs = []
    sentences = [clean_text(x) for x in sentences]

    for start in range(0, len(sentences), batch_size):
        batch_sents = sentences[start:start + batch_size]
        batch_words = [s.split() for s in batch_sents]

        enc = ate_tokenizer(
            batch_words,
            is_split_into_words=True,
            truncation=True,
            padding=True,
            max_length=MAX_LEN_ATE,
            return_tensors='pt'
        )

        word_id_maps = [enc.word_ids(batch_index=i) for i in range(len(batch_words))]
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        logits = ate_model(**enc).logits
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        pred_ids = probs.argmax(axis=-1)
        max_probs = probs.max(axis=-1)

        id2label = ate_model.config.id2label

        for bi, words in enumerate(batch_words):
            word_scores = defaultdict(list)
            word_labels = defaultdict(list)

            for ti, wid in enumerate(word_id_maps[bi]):
                if wid is None:
                    continue
                label_id = int(pred_ids[bi][ti])
                label_name = id2label.get(label_id, str(label_id))
                conf = float(max_probs[bi][ti])
                word_scores[wid].append(conf)
                word_labels[wid].append((label_id, label_name, conf))

            aspects = []
            confs = []
            cur_tokens = []
            cur_confs = []

            for wi, word in enumerate(words):
                if wi not in word_labels:
                    is_asp = False
                    conf = 0.0
                else:
                    # lấy label/conf token đầu hoặc max confident của word
                    best = max(word_labels[wi], key=lambda x: x[2])
                    label_id, label_name, conf = best
                    is_asp = _is_aspect_label(label_name, label_id) and conf >= SPAN_THRESHOLD

                if is_asp:
                    cur_tokens.append(word)
                    cur_confs.append(conf)
                else:
                    if cur_tokens:
                        asp = ' '.join(cur_tokens).strip()
                        aspects.append(asp)
                        confs.append(float(np.mean(cur_confs)))
                        cur_tokens = []
                        cur_confs = []

            if cur_tokens:
                asp = ' '.join(cur_tokens).strip()
                aspects.append(asp)
                confs.append(float(np.mean(cur_confs)))

            # bỏ trùng aspect trong cùng câu, giữ thứ tự
            seen = set()
            uniq_aspects, uniq_confs = [], []
            for a, c in zip(aspects, confs):
                key = a.lower()
                if key not in seen:
                    seen.add(key)
                    uniq_aspects.append(a)
                    uniq_confs.append(c)

            outputs.append((uniq_aspects, uniq_confs))

    return outputs


@torch.no_grad()
def predict_asc_pairs(sentences, aspects, batch_size=BATCH_SIZE_ASC):
    """ASC không dùng threshold, chỉ argmax. Return probs, label_id, confidence."""
    texts = [mark_aspect(s, a) for s, a in zip(sentences, aspects)]
    all_probs, all_labels, all_confs = [], [], []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = asc_tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=MAX_LEN_ASC,
            return_tensors='pt'
        ).to(DEVICE)
        logits = asc_model(**enc).logits
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        labels = probs.argmax(axis=-1)
        confs = probs.max(axis=-1)

        all_probs.extend(probs.tolist())
        all_labels.extend(labels.astype(int).tolist())
        all_confs.extend(confs.astype(float).tolist())

    return all_probs, all_labels, all_confs


In [ ]:

# ============================================================
# 5. PROCESS ONE CATEGORY - có batch checkpoint/resume
# ============================================================


def _batch_file_path(category_name, global_batch_id):
    return P10_BATCH_DIR / f'{category_name}_batch_{global_batch_id:05d}.parquet'


def list_saved_batch_files(category_name):
    return sorted(P10_BATCH_DIR.glob(f'{category_name}_batch_*.parquet'))


def process_one_batch(category_name, batch_df):
    """Chạy Gate -> ATE -> ASC cho 1 batch pandas, trả về pred_df và meta của batch."""
    pred_rows = []

    if len(batch_df) == 0:
        return pd.DataFrame(), {
            'total_sentence_rows': 0,
            'sentence_with_aspect': 0,
            'sentence_multi_aspect': 0,
            'all_review_ids': set(),
            'review_with_aspect': set(),
            'review_multi_aspect': set(),
        }

    if 'sentence_text' not in batch_df.columns:
        raise ValueError(f'{category_name}: thiếu cột sentence_text trong input P2')

    batch_df = batch_df.copy()
    batch_df['sentence_text'] = batch_df['sentence_text'].apply(clean_text)
    batch_df = batch_df[batch_df['sentence_text'].str.len() > 0].reset_index(drop=True)

    if len(batch_df) == 0:
        return pd.DataFrame(), {
            'total_sentence_rows': 0,
            'sentence_with_aspect': 0,
            'sentence_multi_aspect': 0,
            'all_review_ids': set(),
            'review_with_aspect': set(),
            'review_multi_aspect': set(),
        }

    total_sentence_rows = len(batch_df)
    sentence_with_aspect = 0
    sentence_multi_aspect = 0

    all_review_ids = set()
    review_with_aspect = set()
    review_multi_aspect = set()

    review_ids = safe_col(batch_df, 'review_id', default=None).tolist()
    if any(x is not None for x in review_ids):
        all_review_ids.update([x for x in review_ids if x is not None])

    sentences = batch_df['sentence_text'].tolist()
    gate_probs = predict_gate(sentences)

    # chỉ câu có gate >= 0.7 mới chạy ATE
    gate_mask = [p >= GATE_THRESHOLD for p in gate_probs]
    idxs_to_ate = [i for i, ok in enumerate(gate_mask) if ok]
    sents_to_ate = [sentences[i] for i in idxs_to_ate]

    ate_results_map = {i: ([], []) for i in range(len(batch_df))}
    if sents_to_ate:
        ate_results = extract_aspects_batch(sents_to_ate)
        for original_i, result in zip(idxs_to_ate, ate_results):
            ate_results_map[original_i] = result

    # build aspect-level pairs for ASC
    pair_sentence_idx = []
    pair_sentences = []
    pair_aspects = []
    pair_span_conf = []

    for i in range(len(batch_df)):
        aspects, span_confs = ate_results_map[i]
        if len(aspects) > 0:
            sentence_with_aspect += 1
            rid = review_ids[i] if i < len(review_ids) else None
            if rid is not None:
                review_with_aspect.add(rid)
            if len(aspects) > 1:
                sentence_multi_aspect += 1
                if rid is not None:
                    review_multi_aspect.add(rid)

        for a, c in zip(aspects, span_confs):
            pair_sentence_idx.append(i)
            pair_sentences.append(sentences[i])
            pair_aspects.append(a)
            pair_span_conf.append(c)

    if pair_aspects:
        asc_probs, asc_labels, asc_confs = predict_asc_pairs(pair_sentences, pair_aspects)

        parent_asins = safe_col(batch_df, 'parent_asin', default=None).tolist()
        sentence_ids = safe_col(batch_df, 'sentence_id', default=None).tolist()
        ratings = safe_col(batch_df, 'rating', default=None).tolist()

        for j, row_i in enumerate(pair_sentence_idx):
            label_id = int(asc_labels[j])
            pred_rows.append({
                'category_name': category_name,
                'parent_asin': parent_asins[row_i] if row_i < len(parent_asins) else None,
                'review_id': review_ids[row_i] if row_i < len(review_ids) else None,
                'sentence_id': sentence_ids[row_i] if row_i < len(sentence_ids) else None,
                'sentence_text': sentences[row_i],
                'rating': ratings[row_i] if row_i < len(ratings) else None,
                'gate_confidence': float(gate_probs[row_i]),
                'aspect': pair_aspects[j],
                'span_confidence': float(pair_span_conf[j]),
                'sentiment_id': label_id,
                'sentiment': LABEL_NAMES.get(label_id, str(label_id)),
                'asc_confidence': float(asc_confs[j]),
                'prob_neg': float(asc_probs[j][0]),
                'prob_neu': float(asc_probs[j][1]),
                'prob_pos': float(asc_probs[j][2]),
                'sentence_length_chars': len(sentences[row_i]),
            })

    pred_df = pd.DataFrame(pred_rows)
    meta = {
        'total_sentence_rows': int(total_sentence_rows),
        'sentence_with_aspect': int(sentence_with_aspect),
        'sentence_multi_aspect': int(sentence_multi_aspect),
        'all_review_ids': all_review_ids,
        'review_with_aspect': review_with_aspect,
        'review_multi_aspect': review_multi_aspect,
    }
    return pred_df, meta


def process_category(category_name, input_path):
    print('' + '=' * 100)
    print('[CATEGORY]', category_name)
    print('[INPUT]', input_path)
    print('[BATCH CHECKPOINT DIR]', P10_BATCH_DIR)

    total_sentence_rows = 0
    sentence_with_aspect = 0
    sentence_multi_aspect = 0

    all_review_ids = set()
    review_with_aspect = set()
    review_multi_aspect = set()

    # Chạy từng parquet batch. Batch nào đã có file checkpoint thì skip.
    # Có thể chia nhiều tài khoản bằng START_BATCH / END_BATCH.
    global_batch_id = 0
    for batch_df in tqdm(iter_parquet_batches(input_path, PARQUET_BATCH_ROWS), desc=f'{category_name} batches'):
        batch_out_path = _batch_file_path(category_name, global_batch_id)
        meta_out_path = batch_out_path.with_suffix('.meta.json')

        # Chỉ chạy đúng khoảng batch được giao cho tài khoản này.
        if global_batch_id < START_BATCH or global_batch_id > END_BATCH:
            global_batch_id += 1
            continue

        if batch_out_path.exists():
            print(f'[SKIP] batch {global_batch_id} exists -> {batch_out_path.name}')
            # Đọc meta đã lưu để thống kê không bị mất khi resume.
            if meta_out_path.exists():
                try:
                    with open(meta_out_path, 'r', encoding='utf-8') as f:
                        bm = json.load(f)
                    total_sentence_rows += int(bm.get('total_sentence_rows', 0))
                    sentence_with_aspect += int(bm.get('sentence_with_aspect', 0))
                    sentence_multi_aspect += int(bm.get('sentence_multi_aspect', 0))
                    # review ids set không khôi phục đầy đủ để tránh meta quá lớn; final report vẫn có sentence stats chính xác.
                except Exception as e:
                    print('[WARN] Không đọc được meta:', meta_out_path, repr(e))
            global_batch_id += 1
            continue

        try:
            batch_pred_df, bm = process_one_batch(category_name, batch_df)
        except Exception as e:
            print(f'[ERROR] batch {global_batch_id} failed:', repr(e))
            raise

        # Lưu ngay cả batch không có aspect rows để đánh dấu đã xử lý.
        if batch_pred_df.empty:
            batch_pred_df = pd.DataFrame(columns=[
                'category_name', 'parent_asin', 'review_id', 'sentence_id', 'sentence_text', 'rating',
                'gate_confidence', 'aspect', 'span_confidence', 'sentiment_id', 'sentiment',
                'asc_confidence', 'prob_neg', 'prob_neu', 'prob_pos', 'sentence_length_chars'
            ])
        batch_pred_df.to_parquet(batch_out_path, index=False)

        bm_save = {
            'batch_id': int(global_batch_id),
            'total_sentence_rows': int(bm['total_sentence_rows']),
            'sentence_with_aspect': int(bm['sentence_with_aspect']),
            'sentence_multi_aspect': int(bm['sentence_multi_aspect']),
            'aspect_level_rows': int(len(batch_pred_df)),
        }
        with open(meta_out_path, 'w', encoding='utf-8') as f:
            json.dump(bm_save, f, ensure_ascii=False, indent=2)

        total_sentence_rows += int(bm['total_sentence_rows'])
        sentence_with_aspect += int(bm['sentence_with_aspect'])
        sentence_multi_aspect += int(bm['sentence_multi_aspect'])
        all_review_ids.update(bm.get('all_review_ids', set()))
        review_with_aspect.update(bm.get('review_with_aspect', set()))
        review_multi_aspect.update(bm.get('review_multi_aspect', set()))

        print(f'[SAVED] batch {global_batch_id} rows={len(batch_pred_df):,} -> {batch_out_path}')
        global_batch_id += 1

    # Gộp tất cả batch checkpoint đã lưu.
    batch_files = list_saved_batch_files(category_name)
    print(f'[MERGE] Found {len(batch_files)} saved batch files')

    if batch_files:
        dfs = []
        for f in batch_files:
            dfs.append(pd.read_parquet(f))
        pred_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        pred_df = pd.DataFrame()

    pred_path = OUTPUT_DIR / f'final_predictions_{category_name}.parquet'
    pred_df.to_parquet(pred_path, index=False)

    # Backup final lên Drive nếu có.
    try:
        backup_path = P10_FINAL_BACKUP_DIR / pred_path.name
        pred_df.to_parquet(backup_path, index=False)
        print('[BACKUP]', backup_path)
    except Exception as e:
        print('[WARN] Không backup final lên Drive:', repr(e))

    print('[SAVED]', pred_path)
    print('Aspect-level rows:', len(pred_df))

    meta = {
        'total_sentence_rows': int(total_sentence_rows),
        'sentence_with_aspect': int(sentence_with_aspect),
        'sentence_multi_aspect': int(sentence_multi_aspect),
        'total_reviews_if_review_id_exists': int(len(all_review_ids)),
        'reviews_with_aspect_if_review_id_exists': int(len(review_with_aspect)),
        'reviews_multi_aspect_if_review_id_exists': int(len(review_multi_aspect)),
        'saved_batch_files': int(len(batch_files)),
        'batch_checkpoint_dir': str(P10_BATCH_DIR),
    }

    return pred_df, meta, pred_path


In [17]:

# ============================================================
# 6. STATS + REPORT
# ============================================================

def count_rate_table(df, col):
    counts = df[col].value_counts().to_dict()
    total = sum(counts.values())
    return {k: {'count': int(v), 'rate': float(v / total) if total else 0.0} for k, v in counts.items()}


def make_report_for_category(category_name, pred_df, meta):
    aspect_csv_path = OUTPUT_DIR / f'aspect_sentiment_counts_{category_name}.csv'
    neutral_txt_path = OUTPUT_DIR / f'neutral_low_conf_examples_{category_name}.txt'
    report_path = REPORT_DIR / f'p10_report_{category_name}.txt'

    if pred_df.empty:
        aspect_counts = pd.DataFrame(columns=['aspect', 'pos', 'neu', 'neg'])
    else:
        tmp = pred_df.copy()
        tmp['aspect_norm'] = tmp['aspect'].astype(str).str.lower().str.strip()
        pivot = (
            tmp.pivot_table(index='aspect_norm', columns='sentiment', values='sentence_text', aggfunc='count', fill_value=0)
            .reset_index()
            .rename(columns={'aspect_norm': 'aspect'})
        )
        for c in ['pos', 'neu', 'neg']:
            if c not in pivot.columns:
                pivot[c] = 0
        aspect_counts = pivot[['aspect', 'pos', 'neu', 'neg']].copy()
        aspect_counts['total'] = aspect_counts[['pos', 'neu', 'neg']].sum(axis=1)
        aspect_counts = aspect_counts.sort_values('total', ascending=False)

    aspect_counts[['aspect', 'pos', 'neu', 'neg']].to_csv(aspect_csv_path, index=False)

    lines = []
    lines.append('P10 FINAL PIPELINE REPORT')
    lines.append('=' * 80)
    lines.append(f'Category: {category_name}')
    lines.append(f'Gate threshold: {GATE_THRESHOLD}')
    lines.append(f'Span threshold: {SPAN_THRESHOLD}')
    lines.append('ASC threshold: None, use argmax only')
    lines.append('')

    lines.append('[1] Review/Sentence overview')
    lines.append(f"- Sentence rows input: {meta.get('total_sentence_rows', 0):,}")
    lines.append(f"- Sentences with at least 1 aspect: {meta.get('sentence_with_aspect', 0):,}")
    lines.append(f"- Sentences with more than 1 aspect: {meta.get('sentence_multi_aspect', 0):,}")
    if meta.get('total_reviews_if_review_id_exists', 0) > 0:
        lines.append(f"- Reviews input: {meta.get('total_reviews_if_review_id_exists', 0):,}")
        lines.append(f"- Reviews with at least 1 aspect: {meta.get('reviews_with_aspect_if_review_id_exists', 0):,}")
        lines.append(f"- Reviews with more than 1 aspect: {meta.get('reviews_multi_aspect_if_review_id_exists', 0):,}")
    else:
        lines.append('- review_id not found, review-level stats approximated by sentence-level stats only')
    lines.append('')

    lines.append('[2] Sentiment distribution overall')
    if pred_df.empty:
        lines.append('No prediction rows.')
    else:
        total = len(pred_df)
        for sent in ['pos', 'neu', 'neg']:
            cnt = int((pred_df['sentiment'] == sent).sum())
            lines.append(f'- {sent}: {cnt:,} ({cnt / total:.6f})')
    lines.append('')

    lines.append('[3] Sentiment distribution by rating')
    if not pred_df.empty and 'rating' in pred_df.columns:
        rating_tab = pd.crosstab(pred_df['rating'], pred_df['sentiment'])
        lines.append(rating_tab.to_string())
        lines.append('')
        lines.append('Rate by rating:')
        lines.append(pd.crosstab(pred_df['rating'], pred_df['sentiment'], normalize='index').to_string())
    lines.append('')

    lines.append('[4] Top 20 most frequent aspects with pos/neu/neg count')
    if not aspect_counts.empty:
        lines.append(aspect_counts.head(20)[['aspect', 'pos', 'neu', 'neg', 'total']].to_string(index=False))
    lines.append('')

    lines.append('[5] Top 20 most positive aspects by count')
    if not aspect_counts.empty:
        lines.append(aspect_counts.sort_values('pos', ascending=False).head(20)[['aspect', 'pos', 'neu', 'neg']].to_string(index=False))
    lines.append('')

    lines.append('[6] Top 20 most negative aspects by count')
    if not aspect_counts.empty:
        lines.append(aspect_counts.sort_values('neg', ascending=False).head(20)[['aspect', 'pos', 'neu', 'neg']].to_string(index=False))
    lines.append('')

    lines.append('[7] Top 20 divisive aspects')
    # polar >= 10 và abs(pos - neg) nhỏ nhất
    if not aspect_counts.empty:
        div = aspect_counts.copy()
        div['polar'] = div['pos'] + div['neg']
        div['pos_neg_gap'] = (div['pos'] - div['neg']).abs()
        div = div[div['polar'] >= 10].sort_values(['pos_neg_gap', 'polar'], ascending=[True, False]).head(20)
        lines.append(div[['aspect', 'pos', 'neg', 'neu', 'polar', 'pos_neg_gap']].to_string(index=False))
    lines.append('')

    lines.append('[8] Sentence length by sentiment')
    if not pred_df.empty:
        len_stats = pred_df.groupby('sentiment')['sentence_length_chars'].agg(['mean', 'sum', 'count'])
        lines.append(len_stats.to_string())
    lines.append('')

    lines.append('[9] ASC confidence stats by sentiment')
    if not pred_df.empty:
        conf_stats = pred_df.groupby('sentiment')['asc_confidence'].agg(
            mean='mean',
            median='median',
            q1=lambda x: x.quantile(0.25),
            q3=lambda x: x.quantile(0.75),
            count='count'
        )
        lines.append(conf_stats.to_string())
    lines.append('')

    lines.append('[10] Output files')
    lines.append(f'- Aspect sentiment csv: {aspect_csv_path}')
    lines.append(f'- Neutral low confidence examples: {neutral_txt_path}')
    lines.append(f'- Report: {report_path}')

    # 10 câu neutral confidence thấp nhất
    neutral_lines = []
    if not pred_df.empty:
        neu_low = pred_df[pred_df['sentiment'] == 'neu'].sort_values('asc_confidence', ascending=True).head(10)
        for i, row in enumerate(neu_low.itertuples(index=False), start=1):
            neutral_lines.append(f'#{i}')
            neutral_lines.append(f'category: {category_name}')
            neutral_lines.append(f'aspect: {getattr(row, "aspect")}')
            neutral_lines.append(f'asc_confidence: {getattr(row, "asc_confidence"):.6f}')
            neutral_lines.append(f'rating: {getattr(row, "rating")}')
            neutral_lines.append(f'sentence: {getattr(row, "sentence_text")}')
            neutral_lines.append('')

    with open(neutral_txt_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(neutral_lines))

    with open(report_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))

    print('[SAVED]', aspect_csv_path)
    print('[SAVED]', neutral_txt_path)
    print('[SAVED]', report_path)
    return report_path, aspect_csv_path, neutral_txt_path


In [ ]:

# ============================================================
# 7. RUN OFFICE_PRODUCTS ONLY - có thể resume từ batch checkpoint
# ============================================================

all_summary = []

for category_name, input_path in P2_DATA_PATHS.items():
    input_path = Path(input_path)
    if not input_path.exists():
        print(f'[SKIP] Không thấy input cho {category_name}: {input_path}')
        continue

    pred_df, meta, pred_path = process_category(category_name, input_path)
    report_path, aspect_csv_path, neutral_txt_path = make_report_for_category(category_name, pred_df, meta)

    # Backup report/csv/txt sang Drive nếu có.
    try:
        for p in [pred_path, report_path, aspect_csv_path, neutral_txt_path]:
            p = Path(p)
            if p.exists():
                shutil.copy2(p, P10_FINAL_BACKUP_DIR / p.name)
        print('[BACKUP DONE]', P10_FINAL_BACKUP_DIR)
    except Exception as e:
        print('[WARN] Backup output failed:', repr(e))

    all_summary.append({
        'category_name': category_name,
        'input_path': str(input_path),
        'prediction_path': str(pred_path),
        'report_path': str(report_path),
        'aspect_csv_path': str(aspect_csv_path),
        'neutral_examples_path': str(neutral_txt_path),
        **meta,
        'aspect_level_rows': int(len(pred_df)),
    })

summary_path = OUTPUT_DIR / 'p10_office_products_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(all_summary, f, ensure_ascii=False, indent=2)

try:
    shutil.copy2(summary_path, P10_FINAL_BACKUP_DIR / summary_path.name)
except Exception as e:
    print('[WARN] Backup summary failed:', repr(e))

print('DONE')
print('Summary:', summary_path)
print(json.dumps(all_summary, ensure_ascii=False, indent=2)[:3000])


[CATEGORY] office_products
[INPUT] /content/p10_assets/office_products_p2
[BATCH CHECKPOINT DIR] /content/drive/MyDrive/p10_office_batch_checkpoints


office_products batches: 0it [00:00, ?it/s]

[SAVED] batch 190 rows=36,769 -> /content/drive/MyDrive/p10_office_batch_checkpoints/office_products_batch_00190.parquet
[SAVED] batch 191 rows=37,310 -> /content/drive/MyDrive/p10_office_batch_checkpoints/office_products_batch_00191.parquet
[SAVED] batch 192 rows=36,637 -> /content/drive/MyDrive/p10_office_batch_checkpoints/office_products_batch_00192.parquet
[SAVED] batch 193 rows=37,596 -> /content/drive/MyDrive/p10_office_batch_checkpoints/office_products_batch_00193.parquet
[SAVED] batch 194 rows=36,825 -> /content/drive/MyDrive/p10_office_batch_checkpoints/office_products_batch_00194.parquet
[SAVED] batch 195 rows=37,049 -> /content/drive/MyDrive/p10_office_batch_checkpoints/office_products_batch_00195.parquet
[SAVED] batch 196 rows=36,803 -> /content/drive/MyDrive/p10_office_batch_checkpoints/office_products_batch_00196.parquet
[SAVED] batch 197 rows=36,851 -> /content/drive/MyDrive/p10_office_batch_checkpoints/office_products_batch_00197.parquet
[SAVED] batch 198 rows=37,343 ->

In [ ]:

# 8. MERGE ASPECT CSV - OFFICE_PRODUCTS ONLY
# 8. MERGE ASPECT CSV ALL CATEGORIES
# ============================================================

all_csv = []
for category_name in P2_DATA_PATHS.keys():
    p = OUTPUT_DIR / f'aspect_sentiment_counts_{category_name}.csv'
    if p.exists():
        d = pd.read_csv(p)
        d['category_name'] = category_name
        all_csv.append(d)

if all_csv:
    merged = pd.concat(all_csv, ignore_index=True)
    merged_path = OUTPUT_DIR / 'aspect_sentiment_counts_all_categories.csv'
    merged.to_csv(merged_path, index=False)
    print('Saved merged aspect csv:', merged_path)
    display(merged.head())
else:
    print('No category csv found')


In [ ]:

# ============================================================
# 9. CHECK BATCH PROGRESS
# ============================================================

from pathlib import Path
import re

CATEGORY = 'office_products'
files = sorted(P10_BATCH_DIR.glob(f'{CATEGORY}_batch_*.parquet'))
print('Batch checkpoint dir:', P10_BATCH_DIR)
print('Số batch đã lưu:', len(files))
if files:
    print('Batch đầu:', files[0].name)
    print('Batch mới nhất:', files[-1].name)
    print('5 batch mới nhất:')
    for f in files[-5:]:
        print('-', f.name, f.stat().st_size / (1024**2), 'MB')
else:
    print('Chưa có batch checkpoint nào.')

print('START_BATCH/END_BATCH:', START_BATCH, END_BATCH)


In [ ]:

# ============================================================
# 10. ZIP + DOWNLOAD OUTPUT
# ============================================================

from google.colab import files
import shutil
from pathlib import Path

PACK_DIR = Path('/content/p10_office_products_outputs_pack')
ZIP_BASE = '/content/p10_office_products_outputs'
ZIP_PATH = Path(ZIP_BASE + '.zip')

if PACK_DIR.exists():
    shutil.rmtree(PACK_DIR)
PACK_DIR.mkdir(parents=True, exist_ok=True)

# Copy outputs local
if OUTPUT_DIR.exists():
    shutil.copytree(OUTPUT_DIR, PACK_DIR / 'p10', dirs_exist_ok=True)
if REPORT_DIR.exists():
    shutil.copytree(REPORT_DIR, PACK_DIR / 'reports_p10', dirs_exist_ok=True)

# Copy final backup nếu có
if P10_FINAL_BACKUP_DIR.exists():
    shutil.copytree(P10_FINAL_BACKUP_DIR, PACK_DIR / 'drive_final_backup', dirs_exist_ok=True)

# Không zip toàn bộ batch checkpoint vì rất nặng. Chỉ lưu danh sách batch.
batch_list_path = PACK_DIR / 'saved_batch_checkpoints.txt'
with open(batch_list_path, 'w', encoding='utf-8') as f:
    for p in sorted(P10_BATCH_DIR.glob('office_products_batch_*.parquet')):
        f.write(str(p) + '\n')

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
shutil.make_archive(ZIP_BASE, 'zip', root_dir=str(PACK_DIR))

print('Created:', ZIP_PATH)
files.download(str(ZIP_PATH))
